In [0]:
%pip install azure-storage-blob requests

Note: you may need to restart the kernel using dbutils.library.restartPython() to use updated packages.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.9/455.9 kB 6.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.9/220.9 kB 18.5 MB/s eta 0:00:00
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.4.0
    Not uninstalling typing-extensions at /databricks/python3/lib/python3.10/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-ef3e1bfc-9a75-4d9a-bd28-22ba7f6a815f
    Can't uninstall 'typing_extensions'. No files were found to uninstall.
Note: you may need to restart the kernel using dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
import requests, zipfile, os
from concurrent.futures import ThreadPoolExecutor
from azure.storage.blob import BlobServiceClient
import datetime
storage_account = "vesseldatavd99"
storage_key = dbutils.secrets.get(scope="vessel-secrets", key="storage-key")

spark.conf.set(f"fs.azure.account.key.{storage_account}.dfs.core.windows.net", storage_key)

BASE = "https://coast.noaa.gov/htdata/CMSP/AISDataHandler/2019"
START = datetime.date(2019, 1, 1)
END = datetime.date(2019, 1, 29)

svc = BlobServiceClient(account_url=f"https://{storage_account}.blob.core.windows.net", credential=storage_key)
raw = svc.get_container_client("raw")

In [0]:
def fetch(day):
    tag = day.strftime("%Y_%m_%d")
    blob_name = f"zip/AIS_{tag}.zip"
    if raw.get_blob_client(blob_name).exists():
        return f"skip {tag} (already in raw)"

    r = requests.get(f"{BASE}/AIS_{tag}.zip", timeout=900)
    if r.status_code != 200:
        return f"missing {tag}: HTTP {r.status_code}"

    raw.upload_blob(name=blob_name, data=r.content, overwrite=True, timeout=3600)
    return f"fetched {tag} ({len(r.content) / 1e6:.0f} MB)"

MAX_DAYS_PER_RUN = 7
END = datetime.date.today() - datetime.timedelta(days=3)  # NOAA publishes with a lag

all_days = [START + datetime.timedelta(days=i) for i in range((END - START).days + 1)]
missing = [d for d in all_days
           if not raw.get_blob_client(f"zip/AIS_{d:%Y_%m_%d}.zip").exists()]

batch = missing[:MAX_DAYS_PER_RUN]
print(f"missing: {len(missing)}, this run: {len(batch)}")

for day in batch:
    print(fetch(day))

skip 2019_01_01 (already in raw)
skip 2019_01_02 (already in raw)
skip 2019_01_03 (already in raw)
skip 2019_01_04 (already in raw)
skip 2019_01_05 (already in raw)
skip 2019_01_06 (already in raw)
skip 2019_01_07 (already in raw)
skip 2019_01_08 (already in raw)
skip 2019_01_09 (already in raw)
skip 2019_01_10 (already in raw)
skip 2019_01_11 (already in raw)
skip 2019_01_12 (already in raw)
skip 2019_01_13 (already in raw)
skip 2019_01_14 (already in raw)
skip 2019_01_15 (already in raw)
skip 2019_01_16 (already in raw)
fetched 2019_01_17 (254 MB)
fetched 2019_01_18 (254 MB)


In [0]:
for b in raw.list_blobs(name_starts_with="zip/"):
    print(b.name, round(b.size / 1e6), "MB")

zip/AIS_2019_01_01.zip 290 MB
zip/AIS_2019_01_02.zip 276 MB
zip/AIS_2019_01_03.zip 260 MB
zip/AIS_2019_01_04.zip 271 MB
zip/AIS_2019_01_05.zip 267 MB
zip/AIS_2019_01_06.zip 272 MB
zip/AIS_2019_01_07.zip 274 MB
zip/AIS_2019_01_08.zip 277 MB
zip/AIS_2019_01_09.zip 284 MB
zip/AIS_2019_01_10.zip 277 MB
zip/AIS_2019_01_11.zip 262 MB
zip/AIS_2019_01_12.zip 256 MB
zip/AIS_2019_01_13.zip 252 MB
zip/AIS_2019_01_14.zip 263 MB
zip/AIS_2019_01_15.zip 258 MB
zip/AIS_2019_01_16.zip 256 MB
zip/AIS_2019_01_17.zip 254 MB
zip/AIS_2019_01_18.zip 254 MB
